In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [2]:
df = pd.read_csv("malnutrition_children_ethiopia.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
)

df.head()


,id,age_months,gender,region,mother_education,household_wealth_index,height_cm,weight_kg,stunting,underweight,overweight,anemia,malaria,diarrhea,tb,nutrition_status
0,1,31,Male,Amhara,Higher,Middle,84.1,19.2,1,1,0,1,1,1,1,At_Risk
1,2,38,Female,Tigray,Higher,Middle,91.0,6.3,0,0,0,0,1,1,0,Normal
2,3,7,Female,SNNPR,Secondary,Middle,61.4,5.8,1,1,1,1,1,0,1,Normal
3,4,7,Female,Amhara,Higher,Low,103.1,12.9,1,0,1,1,0,1,0,Normal
4,5,0,Male,Tigray,No education,High,78.9,7.4,0,0,0,0,1,1,0,Normal


In [3]:
TARGET = "nutrition_status"

X = df.drop(columns=[TARGET])
y = df[TARGET]

y.value_counts()


nutrition_status
Normal          2030
At_Risk         1238
Malnourished     830
Name: count, dtype: int64

In [4]:
numeric_features = [
    "age_months",
    "height_cm",
    "weight_kg"
]

nominal_features = [
    "gender",
    "region"
]

ordinal_features = [
    "mother_education",
    "household_wealth_index"
]

binary_features = [
    "stunting",
    "underweight",
    "overweight",
    "anemia",
    "malaria",
    "diarrhea",
    "tb"
]


In [5]:
education_order = ["No education", "Primary", "Secondary", "Higher"]
wealth_order = ["Low", "Middle", "High"]

In [6]:
numeric_transformer = StandardScaler()

nominal_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

ordinal_transformer = OrdinalEncoder(
    categories=[education_order, wealth_order]
)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("nom", nominal_transformer, nominal_features),
        ("ord", ordinal_transformer, ordinal_features),
        ("bin", "passthrough", binary_features)
    ]
)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [9]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

nutrition_status
Normal          0.495122
At_Risk         0.302439
Malnourished    0.202439
Name: proportion, dtype: float64

In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_test_processed.shape

((3278, 19), (820, 19))

In [11]:
feature_names = (
    numeric_features
    + list(preprocessor.named_transformers_["nom"].get_feature_names_out(nominal_features))
    + ordinal_features
    + binary_features
)

len(feature_names)

19

In [12]:
X_train_processed.shape[1] == len(feature_names)

True

In [13]:
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

X_train_df.head()

,age_months,height_cm,weight_kg,gender_Female,gender_Male,region_Addis Ababa,region_Amhara,region_Oromia,region_SNNPR,region_Tigray,mother_education,household_wealth_index,stunting,underweight,overweight,anemia,malaria,diarrhea,tb
0,-0.382484,-0.332261,1.226551,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
1,-1.478951,-1.412367,-0.310143,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0
2,-1.248116,-0.373277,-0.424822,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
3,0.483148,0.720501,0.584350,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,2.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0
4,0.771692,-1.166267,0.561415,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0


In [14]:
X_train_df.to_csv("X_train_processed.csv", index=False)
X_test_df.to_csv("X_test_processed.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

In [15]:
df[["age_months", "height_cm", "weight_kg"]].describe()

,age_months,height_cm,weight_kg
count,4098.000000,4098.000000,4098.000000
mean,29.691557,84.532113,12.475915
std,17.357813,14.620986,4.358932
min,0.000000,60.000000,5.000000
25%,15.000000,71.900000,8.700000
50%,29.500000,84.300000,12.400000
75%,45.000000,97.100000,16.300000
max,59.000000,110.000000,20.000000


In [16]:
X_train_df[["age_months", "height_cm", "weight_kg"]].describe()

,age_months,height_cm,weight_kg
count,3.278000e+03,3.278000e+03,3.278000e+03
mean,-1.625708e-18,9.190669e-16,-6.557022e-17
std,1.000153e+00,1.000153e+00,1.000153e+00
min,-1.709786e+00,-1.672139e+00,-1.709223e+00
25%,-8.441541e-01,-8.723137e-01,-8.606008e-01
50%,-3.623085e-02,-1.779934e-02,-1.197864e-02
75%,8.871100e-01,8.640594e-01,8.825150e-01
max,1.695033e+00,1.745918e+00,1.731137e+00


In [17]:
raw = X_train.iloc[0][["age_months","height_cm","weight_kg"]]
scaled = X_train_df.iloc[0][["age_months","height_cm","weight_kg"]]

pd.concat([raw, scaled], axis=1, keys=["raw", "scaled"])

,raw,scaled
age_months,23,-0.382484
height_cm,79.6,-0.332261
weight_kg,17.8,1.226551
